In [2]:
import numpy as np
import pandas as pd
import matplotlib
from synergy.combination import MuSyC # or BRAID, Zimmer
from synergy.combination import Loewe, Bliss # or Bliss, ZIP, HSA, Schindler, CombinationIndex
from matplotlib import pyplot as plt
from synergy.utils import plots
from synergy.utils.dose_tools import grid

ModuleNotFoundError: No module named 'synergy'

### In this notebook we fit MuSyC model for Imi-Clo and Imi-Cypro experiments. 

In [3]:
# This is based on synergy library. 

### Load the data

In [ ]:
imi_clo_reproduction = pd.read_excel("../data/reproduction/Counts_reproduction_IMI_CLO.xlsx")
imi_cypro_reproduction = pd.read_excel("../data/reproduction/Counts_reproduction_IMI_CYPRO.xlsx")

In [4]:
imi_clo_reproduction.columns = ['Rankings', 'Concentrations', 'ToxicUnitsIMI', 'doseIMI', 'ToxicUnitsCLO', 'doseCLO', 'series', 'drug1.conc', 'drug2.conc', 'effect', 'Adults', 'Remarks']
imi_cypro_reproduction.columns = ['ToxicUnits Imidacloprid', 'ToxicUnits Cyproconazole', 'concentration (mg/kg) Imidacloprid', 'TU imi reverse calculated',  'drug2.conc','TU cypro reverse calculated','drug1.conc', 'adults', 'effect']

In [5]:
imi_clo_reproduction['effect'] = imi_clo_reproduction['effect'].astype('float64')
imi_clo_reproduction['drug1.conc'] = imi_clo_reproduction['drug1.conc'].astype('float64')
imi_clo_reproduction['drug2.conc'] = imi_clo_reproduction['drug2.conc'].astype('float64')

In [6]:
imi_cypro_reproduction['effect'] = imi_cypro_reproduction['effect'].astype('float64')
imi_cypro_reproduction['drug1.conc'] = imi_cypro_reproduction['drug1.conc'].astype('float64')
imi_cypro_reproduction['drug2.conc'] = imi_cypro_reproduction['drug2.conc'].astype('float64')

In [7]:
df_exp1 = imi_clo_reproduction[['drug1.conc', 'drug2.conc', 'effect']]
df_exp2 = imi_cypro_reproduction[['drug1.conc', 'drug2.conc', 'effect']]

In [8]:
# Normalize the data
imi_cypro_reproduction['effect'] = (imi_cypro_reproduction['effect']-np.min(imi_cypro_reproduction['effect']))/(np.max(imi_cypro_reproduction['effect'])-np.min(imi_cypro_reproduction['effect']))

In [9]:
# Normalize the data
imi_clo_reproduction['effect'] = (imi_clo_reproduction['effect']-np.min(imi_clo_reproduction['effect']))/(np.max(imi_clo_reproduction['effect'])-np.min(imi_clo_reproduction['effect']))

In [43]:
X_imi_clo = np.concatenate([imi_clo_reproduction['ToxicUnitsIMI'].to_numpy().reshape(-1,1), imi_clo_reproduction['ToxicUnitsCLO'].to_numpy().reshape(-1,1)], axis=1)
Y_imi_clo = imi_clo_reproduction['effect'].to_numpy().reshape(-1,1)
Y_imi_clo = (Y_imi_clo - np.min(Y_imi_clo))/(np.max(Y_imi_clo)-np.min(Y_imi_clo))

In [44]:
X_imi_cypro = np.concatenate([imi_cypro_reproduction['ToxicUnits Imidacloprid'].to_numpy().reshape(-1,1), imi_cypro_reproduction['ToxicUnits Cyproconazole'].to_numpy().reshape(-1,1)], axis=1)
Y_imi_cypro = imi_cypro_reproduction['effect'].to_numpy().reshape(-1,1)
Y_imi_cypro = (Y_imi_cypro-np.min(Y_imi_cypro))/(np.max(Y_imi_cypro)-np.min(Y_imi_cypro))

## Build and fit two models

#### MuSyC for Imi-Clo experiment

In [10]:
model = MuSyC(E1_bounds=(0.0,1.0),E2_bounds=(0.0,1.0), E3_bounds=(0.0,1.0))
# Optimize
model.fit(imi_clo_reproduction['drug1.conc'], imi_clo_reproduction['drug2.conc'], imi_clo_reproduction['effect'], bootstrap_iterations=100)


In [11]:
A_max = np.max(imi_clo_reproduction['drug1.conc'].to_numpy())
B_max = np.max(imi_clo_reproduction['drug2.conc'].to_numpy())

In [12]:
[d11, d22] = np.meshgrid(np.linspace(0.1, A_max, num=100),  np.linspace(0.1, B_max, num=100))
Effect_musyc = model.E(d11, d22)
Effect_musyc = Effect_musyc.reshape((len(d11), len(d22)))


In [13]:
# Get confidence intervals of the parameters
ci = model.get_parameters(confidence_interval=95)

In [14]:
# Parameter estimated and corresponding confidence intervals
ci

{'E0': [0.6574568946485844, array([0.61608064, 0.70629418])],
 'E1': [0.016588421381793853, array([1.30385401e-16, 1.60095429e-01])],
 'E2': [0.03204641279666714, array([1.31815929e-18, 1.20305965e-01])],
 'E3': [0.15335396979754673, array([0.02391678, 0.6153493 ])],
 'h1': [2.895891719290377, array([ 1.81151532, 10.83083304])],
 'h2': [4.270738082846275, array([2.58589845, 8.84710219])],
 'C1': [0.5161799913861921, array([0.34785298, 0.64241273])],
 'C2': [0.2647026170498524, array([0.21784707, 0.33294746])],
 'beta': [-0.213406578917197, array([-9.21081724e-01, -1.70129149e-07])],
 'alpha12': [3.2479492040660567, array([4.28099203e-03, 1.42864302e+05])],
 'alpha21': [3.418511578929687, array([5.77011556e-02, 1.20761210e+02])],
 'gamma12': [0.13558877130464422, array([2.16034124e-04, 3.89659906e+01])],
 'gamma21': [16.402302564197583, array([ 6.57737629, 36.09025578])]}

In [15]:
# Save results into a dataframe
df_ci = pd.DataFrame(np.concatenate([ci['beta'][1].reshape(1,2), ci['alpha12'][1].reshape(1,2), ci['alpha21'][1].reshape(1,2),
ci['gamma12'][1].reshape(1,2), ci['gamma21'][1].reshape(1,2)], axis=0)).round(2)
df_estimate = pd.DataFrame(np.concatenate([ci['beta'][0].reshape(1,1), ci['alpha12'][0].reshape(1,1), ci['alpha21'][0].reshape(1,1),
ci['gamma12'][0].reshape(1,1), ci['gamma21'][0].reshape(1,1)], axis=0)).round(2)
df_result = pd.concat([df_estimate, df_ci], axis=1)
df_result.columns=['estimate', 'ci_l', 'ci_u']
df_result.index=['beta', 'alpha12', 'alpha21', 'gamma12', 'gamma21']

In [16]:
df_result.to_csv('musyc_experiment1.csv')

In [17]:
imi_cypro_reproduction

,ToxicUnits Imidacloprid,ToxicUnits Cyproconazole,concentration (mg/kg) Imidacloprid,TU imi reverse calculated,drug2.conc,TU cypro reverse calculated,drug1.conc,adults,effect
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,0.550943
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,0.671698
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8,0.627358
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,0.675472
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,0.387736
...,...,...,...,...,...,...,...,...,...
135,4.0,4.0,1.6,4.0,1000.0,4.0,1.6,2,0.000000
136,4.0,4.0,1.6,4.0,1000.0,4.0,1.6,8,0.000000
137,4.0,4.0,1.6,4.0,1000.0,4.0,1.6,8,0.000000
138,4.0,4.0,1.6,4.0,1000.0,4.0,1.6,7,0.000000


#### MuSyC for Imi-Cypro experiment

In [18]:
model = MuSyC(E1_bounds=(0.0,1.0),E2_bounds=(0.0,1.0), E3_bounds=(0.0,1.0))
# Optimize
model.fit(imi_cypro_reproduction['drug1.conc'], imi_cypro_reproduction['drug2.conc'], imi_cypro_reproduction['effect'], bootstrap_iterations=100)


In [19]:
A_max = np.max(imi_cypro_reproduction['drug1.conc'].to_numpy())
B_max = np.max(imi_cypro_reproduction['drug2.conc'].to_numpy())

In [20]:
[d11, d22] = np.meshgrid(np.linspace(0.1, A_max, num=100),  np.linspace(0.1, B_max, num=100))
Effect_musyc = model.E(d11, d22)
Effect_musyc = Effect_musyc.reshape((len(d11), len(d22)))


In [21]:
# Get confidence intervals of the parameters
ci = model.get_parameters(confidence_interval=95)


In [22]:
# Parameter estimated and corresponding confidence intervals
ci

{'E0': [0.5543709090350438, array([0.50320764, 0.59250657])],
 'E1': [0.026875109335506145, array([6.52929864e-05, 6.62905256e-02])],
 'E2': [1.4248990597225056e-12, array([5.78969920e-21, 4.71176286e-02])],
 'E3': [7.777547744177824e-08, array([2.73317076e-15, 9.35360987e-01])],
 'h1': [3.6652553000791293, array([2.66142358, 5.51492776])],
 'h2': [2.3916915210861545, array([1.85388179, 3.63579077])],
 'C1': [0.263633861019275, array([0.22261862, 0.31246734])],
 'C2': [168.7281410416616, array([143.76586791, 197.46843031])],
 'beta': [-1.4029244910829553e-07, array([-1.6938338 ,  0.06417137])],
 'alpha12': [0.5024373421492982, array([3.05760289e-05, 3.25533977e+02])],
 'alpha21': [0.24300481472763047, array([0.00121399, 0.7506783 ])],
 'gamma12': [5.490708733302288, array([5.34758108e-08, 1.90495950e+02])],
 'gamma21': [19.79780814561018, array([11.49942429, 82.21360746])]}

In [25]:
df_ci = pd.DataFrame(np.concatenate([ci['beta'][1].reshape(1,2), ci['alpha12'][1].reshape(1,2), ci['alpha21'][1].reshape(1,2),
ci['gamma12'][1].reshape(1,2), ci['gamma21'][1].reshape(1,2)], axis=0)).round(2)
df_estimate = pd.DataFrame(np.concatenate([ci['beta'][0].reshape(1,1), ci['alpha12'][0].reshape(1,1), ci['alpha21'][0].reshape(1,1),
ci['gamma12'][0].reshape(1,1), ci['gamma21'][0].reshape(1,1)], axis=0)).round(2)
df_result = pd.concat([df_estimate, df_ci], axis=1)
df_result.columns=['estimate', 'ci_l', 'ci_u']
df_result.index=['beta', 'alpha12', 'alpha21', 'gamma12', 'gamma21']

In [26]:
df_result.to_csv('musyc_experiment2.csv')